In [1]:
import numpy as np

try:
    import torch
    from torch.nn.utils import parameters_to_vector, vector_to_parameters
except ImportError:
    torch = None
    parameters_to_vector = None
    vector_to_parameters = None


def estimate_rlct_fast(
    f,
    grad_f,
    x0,
    betas=None,
    step_size=1e-3,
    n_steps=4000,
    burn_in=1000,
    thinning=10,
    clip_radius=None,
    grad_clip=None,
    seed=0,
):
    """
    Fast RLCT estimator for high-dimensional fixed f(x) using ULA.

    Target distribution for each beta:
        p_beta(x) ∝ exp(-beta * f(x))

    Estimate:
        lambda ≈ limit_{beta→∞} beta * E_beta[f(X)]

    Strategy:
        1. Run ULA for several betas
        2. Compute y_beta = beta * mean(f(X))
        3. Regress y_beta against 1/log(beta):
               y_beta ≈ lambda + c / log(beta)

    Parameters
    ----------
    f : callable
        f(x) -> scalar
    grad_f : callable
        grad_f(x) -> ndarray same shape as x
    x0 : array-like
        starting point, ideally near the local minimum of interest
    betas : sequence of floats
        inverse temperatures; default is a reasonable geometric ladder
    step_size : float
        Langevin step size
    n_steps : int
        total steps per beta
    burn_in : int
        burn-in steps per beta
    thinning : int
        keep one sample every `thinning` steps after burn-in
    clip_radius : float or None
        if not None, project back to ball ||x-x0|| <= clip_radius
        useful for keeping the chain local around x0
    grad_clip : float or None
        if not None, clip gradient norm to this value
    seed : int
        RNG seed

    Returns
    -------
    result : dict with keys
        lambda_hat : float
        betas : ndarray
        betaEf : ndarray
        mean_f : ndarray
        ess_like_counts : ndarray
    """
    rng = np.random.default_rng(seed)

    x0 = np.asarray(x0, dtype=float)
    dim = x0.size

    if betas is None:
        betas = np.array([8, 16, 32, 64, 128, 256, 512], dtype=float)
    else:
        betas = np.asarray(betas, dtype=float)

    betaEf_list = []
    mean_f_list = []
    counts = []

    # warm start across beta: use last state from previous beta
    x = x0.copy()

    for beta in betas:
        fs = []

        for t in range(n_steps):
            g = np.asarray(grad_f(x), dtype=float)

            if grad_clip is not None:
                gn = np.linalg.norm(g)
                if gn > grad_clip:
                    g = g * (grad_clip / (gn + 1e-12))

            noise = rng.normal(size=dim)
            x = x - step_size * beta * g + np.sqrt(2.0 * step_size) * noise

            # keep the chain local around x0 if desired
            if clip_radius is not None:
                d = x - x0
                nd = np.linalg.norm(d)
                if nd > clip_radius:
                    x = x0 + d * (clip_radius / (nd + 1e-12))

            if t >= burn_in and ((t - burn_in) % thinning == 0):
                fs.append(float(f(x)))

        fs = np.asarray(fs, dtype=float)
        mean_f = fs.mean()
        betaEf = beta * mean_f

        mean_f_list.append(mean_f)
        betaEf_list.append(betaEf)
        counts.append(len(fs))

    mean_f_arr = np.asarray(mean_f_list)
    betaEf_arr = np.asarray(betaEf_list)
    counts = np.asarray(counts)

    # Simple extrapolation: y = lambda + c / log(beta)
    z = 1.0 / np.log(betas)
    A = np.column_stack([np.ones_like(z), z])
    coef, *_ = np.linalg.lstsq(A, betaEf_arr, rcond=None)
    lambda_hat = float(coef[0])

    return {
        "lambda_hat": lambda_hat,
        "betas": betas,
        "betaEf": betaEf_arr,
        "mean_f": mean_f_arr,
        "ess_like_counts": counts,
    }


def make_torch_local_objective(
    model,
    loss_fn,
    x,
    y,
    w0=None,
    loss0=None,
    scale=None,
    device=None,
    dtype=None,
):
    """
    Build NumPy-callable f, grad_f, x0 around a trained PyTorch model.

    The local objective is
        f(w) = scale * (L_n(w) - L_n(w0))
    where L_n is the empirical loss returned by `loss_fn(model, x, y)`.

    Parameters
    ----------
    model : torch.nn.Module
        Trained model whose neighborhood around w0 will be explored.
    loss_fn : callable
        Function of the form loss_fn(model, x, y) -> scalar tensor.
        It should usually return the average empirical loss L_n.
    x, y : torch.Tensor or compatible inputs
        Data used to define the empirical objective.
    w0 : 1D torch.Tensor or np.ndarray, optional
        Reference parameter vector. If None, use the model's current parameters.
    loss0 : float, optional
        Precomputed baseline loss at w0. If None, it is evaluated once.
    scale : float, optional
        Multiplier in front of the shifted loss. Defaults to len(x).
    device : torch.device or str, optional
        Device used for evaluation. Defaults to the model's parameter device.
    dtype : torch.dtype, optional
        Parameter dtype used during evaluation. Defaults to the model dtype.

    Returns
    -------
    f : callable
        NumPy-callable scalar objective.
    grad_f : callable
        NumPy-callable gradient.
    x0_np : ndarray
        Flattened reference parameter vector.
    info : dict
        Contains loss0, scale, device, and dtype.
    """
    if torch is None:
        raise ImportError("PyTorch is required for make_torch_local_objective.")

    params = [p for p in model.parameters() if p.requires_grad]
    if not params:
        raise ValueError("Model has no trainable parameters.")

    ref_param = params[0]
    if device is None:
        device = ref_param.device
    if dtype is None:
        dtype = ref_param.dtype

    model = model.to(device=device, dtype=dtype)
    x = x.to(device=device)
    y = y.to(device=device)

    if scale is None:
        try:
            scale = float(len(x))
        except TypeError as exc:
            raise ValueError("Please provide scale explicitly when len(x) is unavailable.") from exc
    else:
        scale = float(scale)

    if w0 is None:
        with torch.no_grad():
            w0_torch = parameters_to_vector([p.detach() for p in params]).to(device=device, dtype=dtype)
    else:
        w0_torch = torch.as_tensor(w0, device=device, dtype=dtype).reshape(-1)

    num_params = sum(p.numel() for p in params)
    if w0_torch.numel() != num_params:
        raise ValueError(f"w0 has size {w0_torch.numel()}, but model expects {num_params} parameters.")

    def _set_params_from_vector(w_vec):
        with torch.no_grad():
            vector_to_parameters(w_vec, params)

    _set_params_from_vector(w0_torch)

    if loss0 is None:
        with torch.no_grad():
            loss0_value = float(loss_fn(model, x, y).detach().cpu().item())
    else:
        loss0_value = float(loss0)

    x0_np = w0_torch.detach().cpu().double().numpy().copy()

    def f(w_np):
        w_vec = torch.as_tensor(w_np, device=device, dtype=dtype).reshape(-1)
        _set_params_from_vector(w_vec)
        with torch.no_grad():
            loss_value = float(loss_fn(model, x, y).detach().cpu().item())
        return scale * (loss_value - loss0_value)

    def grad_f(w_np):
        w_vec = torch.as_tensor(w_np, device=device, dtype=dtype).reshape(-1)
        _set_params_from_vector(w_vec)
        for p in params:
            if p.grad is not None:
                p.grad.zero_()
        loss_tensor = loss_fn(model, x, y)
        obj = scale * (loss_tensor - loss0_value)
        grads = torch.autograd.grad(obj, params, allow_unused=False)
        grad_vec = parameters_to_vector([g.detach() for g in grads])
        return grad_vec.cpu().double().numpy()

    info = {
        "loss0": loss0_value,
        "scale": scale,
        "device": str(device),
        "dtype": str(dtype),
    }
    return f, grad_f, x0_np, info


def estimate_rlct_fast_torch_local(
    model,
    loss_fn,
    x,
    y,
    w0=None,
    loss0=None,
    scale=None,
    device=None,
    dtype=None,
    **rlct_kwargs,
):
    """
    Convenience wrapper for local RLCT estimation around a PyTorch model.

    Example
    -------
    result = estimate_rlct_fast_torch_local(
        model,
        lambda m, xb, yb: criterion(m(xb), yb),
        x_train,
        y_train,
        clip_radius=0.1,
        betas=[8, 16, 32, 64],
        step_size=1e-5,
    )
    """
    f, grad_f, x0, info = make_torch_local_objective(
        model=model,
        loss_fn=loss_fn,
        x=x,
        y=y,
        w0=w0,
        loss0=loss0,
        scale=scale,
        device=device,
        dtype=dtype,
    )
    result = estimate_rlct_fast(f=f, grad_f=grad_f, x0=x0, **rlct_kwargs)
    result["x0"] = x0
    result["objective_info"] = info
    return result


if __name__ == "__main__":
    # -----------------------------
    # Example 1: isotropic quadratic
    # f(x) = ||x||^2
    # RLCT = d/2
    # -----------------------------
    d = 1000
    x0 = np.zeros(d)

    def f_quad(x):
        return np.dot(x, x)

    def grad_f_quad(x):
        return 2.0 * x

    out = estimate_rlct_fast(
        f=f_quad,
        grad_f=grad_f_quad,
        x0=x0,
        betas=[4, 8, 16, 32, 64, 128, 256],
        step_size=5e-4,
        n_steps=5000,
        burn_in=1000,
        thinning=10,
        clip_radius=2.0,
        grad_clip=100.0,
        seed=0,
    )

    print("=== quadratic example ===")
    print("dimension d =", d)
    print("theory lambda =", d / 2)
    print("estimated lambda =", out["lambda_hat"])
    print("beta * E_beta[f] =", out["betaEf"])

    # -----------------------------
    # Example 2: anisotropic quadratic
    # f(x) = sum a_i x_i^2
    # RLCT is still d/2 if non-singular
    # -----------------------------
    a = np.linspace(0.5, 3.0, d)

    def f_aniso(x):
        return np.sum(a * x * x)

    def grad_f_aniso(x):
        return 2.0 * a * x

    out2 = estimate_rlct_fast(
        f=f_aniso,
        grad_f=grad_f_aniso,
        x0=x0,
        betas=[4, 8, 16, 32, 64, 128, 256],
        step_size=3e-4,
        n_steps=10000,
        burn_in=1000,
        thinning=10,
        clip_radius=2.0,
        grad_clip=100.0,
        seed=1,
    )

    print("\n=== anisotropic quadratic example ===")
    print("theory lambda =", d / 2)
    print("estimated lambda =", out2["lambda_hat"])
    print("beta * E_beta[f] =", out2["betaEf"])

=== quadratic example ===
dimension d = 1000
theory lambda = 500.0
estimated lambda = 549.3774168526589
beta * E_beta[f] = [ 16.          32.          64.         128.         256.
 505.71437204 574.53188578]

=== anisotropic quadratic example ===
theory lambda = 500.0
estimated lambda = 624.7827508905647
beta * E_beta[f] = [ 27.74833198  54.98866727 107.73911413 207.56988567 384.78480447
 537.40919762 580.60662031]


In [2]:
from test_sgld_torch_local import run_tiny_torch_local_test

torch_test_result = run_tiny_torch_local_test(estimate_rlct_fast_torch_local)
torch_test_result


train_loss=0.000577
lambda_hat=1.197364
betaEf=[1.33068213 0.80081084 1.46260907]
scale=48.0
num_params=33


{'lambda_hat': 1.1973643559651508,
 'betas': array([ 4.,  8., 16.]),
 'betaEf': array([1.33068213, 0.80081084, 1.46260907]),
 'mean_f': array([0.33267053, 0.10010135, 0.09141307]),
 'ess_like_counts': array([30, 30, 30]),
 'x0': array([-0.16662379,  0.52357596, -1.20683883,  0.3600811 , -1.05791788,
        -0.12837631, -0.48195241,  0.44792814,  0.07426185,  0.68328311,
        -0.16809566, -0.74314411, -0.28031529,  0.22594369, -0.33050603,
         0.39310474, -0.16216352, -0.74479821,  1.10079813, -1.26160641,
        -1.86042667, -0.67941755,  0.93445692, -0.85674711, -0.84515513,
        -0.24195309, -0.43874522, -0.92345401, -0.68284819,  0.890664  ,
        -0.50948941, -0.57041354, -0.87413113]),
 'objective_info': {'loss0': 0.0005766130068613369,
  'scale': 48.0,
  'device': 'cpu',
  'dtype': 'torch.float64'}}